## Test a New Strategy

In [1]:
from QRT_utils import *
from local_data.data import *
from local_data.constants import *

### Raw Data

In [2]:
historical_bb = pd.read_parquet(os.path.join(PRICE_DIR, BB_HISTORICAL))
active_lseg = pd.read_parquet(os.path.join(PRICE_DIR, LSEG_ACTIVE))

### Active Constituents Data

In [3]:
active_prices_spx, active_vol_spx = get_timeseries(active_lseg, market='.SPX'), get_timeseries(active_lseg, value='Volume', market='.SPX')
active_eligible_spx = eligible_to_trade(active_prices_spx, active_vol_spx, market='.SPX')

active_prices_stoxx, active_vol_stoxx = get_timeseries(active_lseg, market='.STOXX50E'), get_timeseries(active_lseg, value='Volume', market='.STOXX50E')
active_eligible_stoxx = eligible_to_trade(active_prices_stoxx, active_vol_stoxx, market='.STOXX50E')

### Historical Constituents Data

In [4]:
hist_prices_spx, hist_vol_spx = get_timeseries(historical_bb, market='.SPX', data_type='historical'), get_timeseries(historical_bb, value='Volume', market='.SPX', data_type='historical')
hist_eligible_spx = eligible_to_trade(hist_prices_spx, hist_vol_spx, market='.SPX')

hist_prices_stoxx, hist_vol_stoxx = get_timeseries(historical_bb, market='.STOXX50E', data_type='historical'), get_timeseries(historical_bb, value='Volume', market='.STOXX50E', data_type='historical')
hist_eligible_stoxx = eligible_to_trade(hist_prices_stoxx, hist_vol_stoxx, market='.STOXX50E')

### 1. Run Strategy: Momentum Long-Short

In [ ]:
import cvxpy as cp

EXECUTION_COST_BPS = 0.0002
FINANCING_COST_ANNUAL = 0.005

def momentum_portfolio(
    price_data: pd.DataFrame,
    vol_eligible: pd.DataFrame,
    portfolio_start: str = '2026-01-01',

    lookback: int = 252,
    skip_last: int = 21,
    vol_quantile: float = 0.90,
    mom_quantile_top: float = 0.85,
    mom_quantile_bottom: float = 0.15,
    target_vol: float = 0.10,
    weight_max: float = 0.10,
    weight_min: float = 0.005,
    # target_risk: float = 300_000.0
):
    """
    Optimize a beta-neutral momentum portfolio with volatility filtering.

    Args:
        price_data:          Full price DataFrame with tickers as columns, including '.SPX'.
        vol_eligible:        Truth DataFrame for dates when the $ADV ≥ 5M.
        lookback:            Number of trading days of history used to estimate momentum,
                             vol, covariance, and betas. Training window ends the day
                             before portfolio_start.
        portfolio_start:     First date of the live portfolio. The closing price on this
                             date is the entry price. Training uses [portfolio_start -
                             lookback days, portfolio_start).
        vol_quantile:        Max volatility percentile filter.
        mom_quantile_top:    Top momentum percentile to go long.
        mom_quantile_bottom: Bottom momentum percentile to go short.
        target_vol:          Max annualized portfolio volatility.
        weight_max:          Max absolute weight per stock.
        weight_min:          Min absolute weight per stock.
        target_risk:         Dollar notional scalar for position sizing.

    Returns:
        positions: pd.Series of dollar positions (rounded), indexed by ticker.
        stats:     dict with expected return, vol, sharpe, beta, gross leverage,
                   universe size, and entry_date.
    """

    # --- Locate entry date in index ---
    all_dates = price_data.index
    start_idx = all_dates.searchsorted(pd.Timestamp(portfolio_start))
    
    if start_idx >= len(all_dates): # portfolio_start after most recent data
        entry_date = pd.Timestamp(portfolio_start)
    else:
        entry_date = all_dates[start_idx] # actual trading day from data
    
    # --- Training window: lookback days ending strictly before entry ---
    if start_idx < lookback:
        raise ValueError(
            f"Not enough history before {entry_date}: need {lookback} days, "
            f"have {start_idx}."
        )

    # Recent data required to create the portfolio at the current moment of time 'entry_date'
    train_prices = price_data.iloc[start_idx - lookback : start_idx]
    # Fill up to 2 consecutive missing days before calculating returns and removing cols with any nulls
    returns      = train_prices.ffill(limit=2).pct_change(fill_method=None).iloc[1:].dropna(axis=1, how='any')

    # --- Volatility filter ---
    ann_vol    = returns.std() * np.sqrt(252)
    vol_filter = ann_vol[ann_vol < ann_vol.quantile(vol_quantile)].index

    # --- Momentum filter (point-in-time: first to last of training window) ---
    momentum   = (train_prices.iloc[-1-skip_last] / train_prices.ffill().iloc[0]) - 1
    top        = momentum[momentum >= momentum.quantile(mom_quantile_top)].index
    bottom     = momentum[momentum <= momentum.quantile(mom_quantile_bottom)].index
    mom_filter = top.union(bottom)

    # --- Universe --- Vol checks out, Mom checks out, $ADV checks out, active lseg ric's
    vol_eligible_tickers = vol_eligible.columns[vol_eligible.loc[entry_date]]

    # print('vol_filter', len(vol_filter.to_list()))
    # print('mom_filter', len(mom_filter.to_list()))
    # print('vol_eligible_tickers', len(vol_eligible_tickers.to_list()))
    universe = vol_filter.intersection(mom_filter).intersection(vol_eligible_tickers)
    
    if len(universe) == 0:
        raise ValueError("No stocks passed the filters. Adjust parameters.")

    returns_uni = returns[universe]
    mkt_index   = returns.iloc[:,0].ffill(limit=2)
    valid_idx   = mkt_index.dropna().index.intersection(returns_uni.index)

    returns_uni = returns_uni.loc[valid_idx]
    mkt_index   = mkt_index.loc[valid_idx]

    # --- Stats ---
    mu  = returns_uni.mean() * 252
    cov = returns_uni.cov() * 252
    cov += np.eye(len(cov)) * 1e-6
    b   = returns_uni.apply(lambda c: np.cov(c, mkt_index)[0, 1] / np.var(mkt_index))
    if np.var(mkt_index) == 0:
        raise ZeroDivisionError("Market index has zero variance, check data again")

    # --- Optimisation ---
    w    = cp.Variable(len(universe))
    prob = cp.Problem(
        cp.Maximize(mu.values @ w),
        [ # Portfolio constraints
            cp.quad_form(w, cov.values) <= target_vol ** 2, # Variance: w'Σw <= σ^2
            b.values @ w == 0,  # Beta neturality: β'w = 0
            cp.sum(w) == 0,     # Weights sum to zero
            cp.norm(w, 1) <= 2, # Normalise weights
            w >= -weight_max,   # Biggest short weight position
            w <= weight_max,    # Biggest long weight position
        ]
    )

    # prob.solve(solver=cp.ECOS, verbose=False)
    prob.solve(solver=cp.SCS, verbose=False)

    if prob.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
        raise RuntimeError(f"Solver failed: {prob.status}")

    weights   = pd.Series(w.value, index=universe)
    weights   = weights[abs(weights) >= weight_min]
    # Remove stocks which are selected but don't meet min weight threshold
    mu, cov, b = mu[weights.index], cov.loc[weights.index, weights.index], b[weights.index]

    # 252-day in-sample vol for reporting (matches optimizer)
    insample_vol = float(np.sqrt(weights @ cov @ weights))

    # 60-day vol for position sizing (matches risk()) using prev 60 trading days
    # prev_60d_returns = returns_uni[weights.index].tail(60)

    # daily_pnl_unit = prev_60d_returns @ weights
    # unit_risk = daily_pnl_unit.std(ddof=1) * np.sqrt(252)
    # scale = target_risk / unit_risk
    # positions = weights * scale

    # dollar_risk = (prev_60d_returns @ positions).std(ddof=1) * np.sqrt(252)

    # gross_notional = int(positions.abs().sum())  # in dollars

    # exec_cost_dollar    = gross_notional * EXECUTION_COST_BPS  # 2 bps on gross
    # financing_cost_dollar = gross_notional * FINANCING_COST_ANNUAL / 360 # 0.5% on GMV

    stats = {
        'In-Sample Return':       f'{mu @ weights:.2%}',
        'In-Sample Vol':          f'{insample_vol:.2%}',
        # 'Risk':                   f'${dollar_risk:,.0f}',
        'Beta':                   f'{(b * weights).sum():.3f}',
        # 'Universe Size':          len(positions),
        'Longs':                  len(weights[weights>0]),
        'Shorts':                 len(weights[weights<0]),
        'Gross Leverage':         f'{weights.abs().sum():.2f}x',
        # 'Gross Notional':         f'${gross_notional:,.0f}',
        # 'Long Notational':        f'${int(positions[positions>0].abs().sum()):,.0f}',
        # 'Short Notational':       f'${int(positions[positions<0].abs().sum()):,.0f}',
        # 'Exec Cost ($)':          f'${exec_cost_dollar:,.0f}',
        # 'Financing Cost ($/day)': f'${financing_cost_dollar:,.0f}',
    }

    return weights, stats

start_date = '2026-03-31'
positions_spx, stats_spx = momentum_portfolio(
    price_data=active_prices_spx, 
    vol_eligible=active_eligible_spx, 
    portfolio_start=start_date,
    lookback=252, skip_last=21, target_risk=500_000, mom_quantile_top=0.9, mom_quantile_bottom=0.1
)
print(pd.Series(stats_spx))

positions_stoxx, stats_stoxx = momentum_portfolio(
    price_data=active_prices_stoxx, 
    vol_eligible=active_eligible_stoxx,
    portfolio_start=start_date,
    lookback=252, skip_last=21, target_risk=500_000, mom_quantile_top=0.9, mom_quantile_bottom=0.1
)
print(pd.Series(stats_stoxx))

In-Sample Return             159.62%
In-Sample Vol                  9.72%
Risk                        $500,000
Beta                           0.007
Universe Size                     80
Longs                             42
Shorts                            38
Gross Leverage                 1.95x
Gross Notional            $8,816,007
Long Notational           $4,418,129
Short Notational          $4,397,877
Exec Cost ($)                 $1,763
Financing Cost ($/day)          $122
dtype: object
In-Sample Return             103.02%
In-Sample Vol                  9.98%
Risk                        $500,000
Beta                           0.001
Universe Size                     42
Longs                             20
Shorts                            22
Gross Leverage                 2.00x
Gross Notional            $7,708,871
Long Notational           $3,858,416
Short Notational          $3,850,454
Exec Cost ($)                 $1,542
Financing Cost ($/day)          $107
dtype: object


### 2. Backtest with all historical data

In [19]:
def screen_event_impact(
    positions: pd.Series,
    price_data: pd.DataFrame,
    market: Literal['.SPX', '.STOXX50E'],
    reb_date: pd.Timestamp = None,
    days: int = 20,
    z_threshold: float = 1.0,
    plot: bool = True,
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Screen positions for short-term price moves that work against you
    using a z-score approach: flag positions whose adverse return is
    more than z_threshold standard deviations worse than their same-side
    peers.  Removed notional is redistributed equally across survivors,
    then the portfolio is rescaled to match the original risk level,
    and a market hedge is added to target beta ≈ 0.

    Parameters
    ----------
    positions   : pd.Series  – RIC → signed notional (positive = long)
    market      : str         – benchmark RIC for beta calc
    days        : int         – look-back window
    z_threshold : float       – how many σ worse than same-side mean to flag
    plot        : bool        – show normalised-price chart

    Returns
    -------
    flagged    : DataFrame of flagged positions sorted worst-first
    reweighted : pd.Series with reversals removed, capital redistributed,
                 risk matched, and beta ≈ 0
    """
    if reb_date is None:
        reb_date = price_data.index[-1]
    
    ret_col = f"{days}d_return_pct"
    ric_data: dict[str, dict] = {}
    norm_series: dict[str, tuple[pd.Series, str]] = {}

    # ---- pass 1: collect returns for every position ----
    for ric, notional in positions.items():

        close = price_data[ric].loc[:reb_date].iloc[:-1].dropna().tail(days)
        if close.empty or len(close) < days:
            continue

        ret = close.iloc[-1] / close.iloc[0] - 1
        side = "long" if notional > 0 else "short"
        pain = -ret if side == "long" else ret

        norm_series[ric] = (close / close.iloc[0] * 100, side)
        ric_data[ric] = {
            "notional": notional,
            "side": side,
            "ret": ret,
            "pain": pain,
        }

    # ---- pass 2: z-score within each side ----
    rows = []
    for side_label in ("long", "short"):
        peers = {r: d for r, d in ric_data.items() if d["side"] == side_label}
        if len(peers) < 2:
            continue
        pains = np.array([d["pain"] for d in peers.values()])
        mu, sigma = pains.mean(), pains.std(ddof=1)
        if sigma < 1e-9:
            continue

        for ric, d in peers.items():
            z = (d["pain"] - mu) / sigma
            if z > z_threshold:
                rows.append({
                    "RIC": ric,
                    "notional": d["notional"],
                    "side": side_label,
                    ret_col: round(d["ret"] * 100, 2),
                    "z_score": round(z, 2),
                    "action": "reduce / exit",
                })

    flagged = (
        pd.DataFrame(rows)
        .sort_values("z_score", ascending=False)
        .reset_index(drop=True)
    )

    # ---- build reweighted positions ----
    original_risk = risk(positions)

    flagged_rics = set(flagged["RIC"].values) if not flagged.empty else set()
    flagged_idx = list(set(flagged_rics).intersection(positions.index))
    remaining = positions.drop(index=flagged_idx)

    if remaining.empty:
        return flagged, positions
    else:
        longs = remaining[remaining > 0]
        shorts = remaining[remaining < 0]

        flagged_pos = positions.loc[flagged_idx]
        freed_long = flagged_pos[flagged_pos > 0].sum()
        freed_short = flagged_pos[flagged_pos < 0].sum()

        if not longs.empty:
            remaining.loc[longs.index] += freed_long / len(longs)
        if not shorts.empty:
            remaining.loc[shorts.index] += freed_short / len(shorts)

    reweighted = remaining.copy()

    new_risk = risk(reweighted)
    if new_risk > 0:
        reweighted *= original_risk / new_risk

    # hedge to beta ≈ 0
    hedge = forced_hedge(reweighted, market)
    if abs(hedge) > 0.01:
        reweighted[market] = reweighted.get(market, 0.0) + hedge

    # ---- plot ----
    if plot and norm_series:
        fig, (ax_l, ax_s) = plt.subplots(1, 2, figsize=(14, 5))
        for ric, (series, side) in norm_series.items():
            ax = ax_l if side == "long" else ax_s
            style = ({"linewidth": 2, "alpha": 0.9}
                     if ric in flagged_rics else {"alpha": 0.35})
            ax.plot(series, label=ric, **style)

        for ax, title in [(ax_l, "Longs"), (ax_s, "Shorts")]:
            ax.set_title(title)
            ax.axhline(100, color="black", ls="--", lw=0.5)
            ax.set_ylabel("Normalised Price")
            ax.tick_params(axis="x", rotation=45)
            if ax.has_data():
                ax.legend(fontsize=7, loc="best")

        fig.suptitle(
            f"Last {days}d — z>{z_threshold} flagged positions highlighted",
            fontsize=11,
        )
        plt.tight_layout()
        plt.show()

    return flagged, reweighted

def backtest_momentum(
    price_data: pd.DataFrame,
    vol_eligible: pd.DataFrame,
    mkt_index: Literal['.SPX', '.STOXX50E'],
    start_date: str = '2026-01-01',
    end_date: str = None,
    rebalance_freq: int = 10,
    lookback: int = 252,
    target_risk: float = 500_000,
    initial_capital: float = 10_000_000,
    exec_bps: float = 2.0,
    financing_annual: float = 0.005,
    **kwargs
):
    all_dates = price_data.index
    start_idx = all_dates.searchsorted(pd.Timestamp(start_date))
    end_idx = len(all_dates) if end_date is None else all_dates.searchsorted(pd.Timestamp(end_date))

    rebal_indices = list(range(start_idx, end_idx, rebalance_freq))

    daily_ret = pd.Series(0.0, index=all_dates[start_idx:end_idx], dtype=float)
    turnovers = []
    prev_positions = pd.Series(dtype=float)
    nav = initial_capital

    for i, reb_idx in enumerate(rebal_indices):
        reb_date = all_dates[reb_idx]

        if i + 1 < len(rebal_indices):
            next_idx = rebal_indices[i + 1]
        else:
            next_idx = end_idx

        try:
            positions, stats = momentum_portfolio(
                price_data, vol_eligible,
                mkt_index=mkt_index,
                lookback=lookback,
                portfolio_start=str(reb_date.date()),
                target_risk=target_risk,
                **kwargs
            )

            flagged, positions = screen_event_impact(
                positions,
                price_data=price_data,
                reb_date=reb_date,
                market=mkt_index,
                days=20,
                z_threshold=1.0,
                plot=False
            )
        except (ValueError, RuntimeError) as e:
            print(f"[{reb_date.date()}] Skipping: {e}")
            continue

        # turnover
        all_tickers = positions.index.union(prev_positions.index)
        old = prev_positions.reindex(all_tickers, fill_value=0)
        new = positions.reindex(all_tickers, fill_value=0)
        turnover = (new.drop(mkt_index, errors='ignore') - old.drop(mkt_index, errors='ignore')).abs().sum()
        turnovers.append({
            'date': reb_date, 'turnover': turnover,
            'gross_notional': positions.abs().sum(),
            'n_stocks': len(positions), **stats
        })

        # hold period dollar P&L
        hold_prices = price_data.iloc[reb_idx:next_idx][positions.index]
        
        tradable_positions = positions.drop(index=mkt_index, errors='ignore')
        hold_prices = price_data.iloc[reb_idx:next_idx][tradable_positions.index]

        entry_prices = hold_prices.iloc[0]
        shares = tradable_positions / entry_prices
        hold_pnl = (hold_prices.diff() * shares).sum(axis=1).fillna(0)

        # financing & execution (dollar)
        short_notional = (hold_prices * shares).where(shares < 0, 0).abs().sum(axis=1)
        daily_financing = short_notional * financing_annual / 252
        hold_pnl -= daily_financing

        exec_cost = turnover * (exec_bps / 1e4)
        hold_pnl.iloc[0] -= exec_cost

        # convert to percent returns off running NAV
        for dt, dollar_pnl in hold_pnl.items():
            daily_ret.loc[dt] = dollar_pnl / nav
            nav *= (1 + daily_ret.loc[dt])

        # drifted positions
        end_prices = hold_prices.iloc[-1]
        prev_positions = shares * end_prices

    # summary
    cum_ret = (1 + daily_ret).cumprod() - 1
    ann_ret = daily_ret.mean() * 252
    ann_vol = daily_ret.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0

    print(f"Backtest: {daily_ret.index[0].date()} → {daily_ret.index[-1].date()}")
    print(f"Cumulative Return: {cum_ret.iloc[-1]:.2%}")
    print(f"Ann. Return:       {ann_ret:.2%}")
    print(f"Ann. Vol:          {ann_vol:.2%}")
    print(f"Sharpe:            {sharpe:.2f}")
    print(f"Rebalances:        {len(turnovers)}")

    return daily_ret, turnovers

ret, rebs = backtest_momentum(
    prices_df_spx, vol_eligible_spx,
    mkt_index='.SPX',
    start_date='2002-03-20',
    end_date='2026-03-20',
    rebalance_freq=7,
    lookback=252,
    target_risk=500_000,
    initial_capital=10_000_000
)

((1 + ret).cumprod() - 1).plot(title='Cumulative Return (%)')

AttributeError: 'NAType' object has no attribute 'round'

### Screen Portfolios for Reversals

In [21]:

new_pos_portfolios = []
for pos, mkt, prc in [(positions_stoxx, '.STOXX50E', prices_df_stoxx), (positions_spx, '.SPX', prices_df_spx)]:
    flagged, new_positions = screen_event_impact(
        pos, price_data=prc, market=mkt, days=20, z_threshold=1.0, plot=False
    )

    display(flagged)
    print(f"\nOld risk: {risk(pos):,}")
    print(f"New risk: {risk(new_positions):,}")
    print(f"Old beta: {portfolio_beta(pos, mkt):.4f}")
    print(f"New beta: {portfolio_beta(new_positions, mkt):.4f}")
    
    new_pos_portfolios.append(new_positions)


,RIC,notional,side,20d_return_pct,z_score,action
0,IPO-SOFT.L,-32506.761427,short,5.02,2.22,reduce / exit
1,MNDI.L,-65085.128172,short,1.96,1.67,reduce / exit
2,SAND.ST,42147.990612,long,-8.71,1.48,reduce / exit
3,HIK.L,-162433.589466,short,0.80,1.47,reduce / exit
4,TUB.BR,137848.138673,long,-5.49,1.04,reduce / exit
5,MAERSKb.CO,85351.763994,long,-5.30,1.02,reduce / exit



Old risk: 508,333
New risk: 508,535
Old beta: 0.0076
New beta: 0.0000


,RIC,notional,side,20d_return_pct,z_score,action
0,VSCO.N,72255.646647,long,-30.31,2.06,reduce / exit
1,VICR.OQ,42184.664314,long,-29.58,2.01,reduce / exit
2,KTOS.OQ,76228.609630,long,-26.61,1.81,reduce / exit
3,TDOC.N,-23344.642205,short,0.78,1.76,reduce / exit
4,HPQ.N,-184491.700130,short,0.21,1.70,reduce / exit
5,BWIN.OQ,-24997.896643,short,-1.26,1.54,reduce / exit
6,HLF.N,91618.453305,long,-18.77,1.26,reduce / exit
7,FIGS.N,42563.773293,long,-15.04,1.01,reduce / exit



Old risk: 489,626
New risk: 488,960
Old beta: 0.0003
New beta: -0.0000


### Scale Risk to 500k USD

In [ ]:
target_risk_usd = 500_000
eurusd = eur_usd()

combined_portfolios = pd.concat([new_pos_portfolios[1], new_pos_portfolios[0] * eurusd])
combined_usd_risk = risk(combined_portfolios)
scale = target_risk_usd / combined_usd_risk

scaled_spx = (new_pos_portfolios[1] * scale).round(2)
scaled_stoxx = (new_pos_portfolios[0] * scale).round(2)

combined_scaled = pd.concat([scaled_spx, scaled_stoxx * eurusd])
print("Scale:", round(scale, 2))
print(f"Scaled combined risk (USD): {risk(combined_scaled):,}")
print("SPX risk:", risk(scaled_spx))
print("STOXX risk:", risk(scaled_stoxx))

Scale: 0.58
Scaled combined risk (USD): 500,000
SPX risk: 283714
STOXX risk: 295073


### Plot Recent Performance

In [ ]:
start_date = '2026-03-31'
for pos, mkt in [(scaled_stoxx, '.STOXX50E'), (scaled_spx, '.SPX')]:
    plot_portfolio_returns(pos, market=mkt, start_date=start_date, figsize=(10,3))

### Submit Positions to QRT

In [11]:
positions_stoxx

targets = (
        positions_stoxx
        .rename_axis('internal_code')
        .reset_index(name='target_notional')
        .assign(currency='EUR')
        .sort_values('target_notional', ascending=False, ignore_index=True)
    )
targets

,internal_code,target_notional,currency
0,SRP.L,386018.263053,EUR
1,SUBC.OL,386017.802090,EUR
2,HUBN.S,386017.525811,EUR
3,BALF.L,366803.916411,EUR
4,AAF.L,286557.194224,EUR
5,TLIT.MI,262613.430675,EUR
6,SDR.L,249032.134205,EUR
7,JYSK.CO,186536.910415,EUR
8,IMCD.AS,185349.096428,EUR
9,GLEN.L,170567.361802,EUR


In [24]:
for positions, region, currency in [(scaled_spx, 'AMER', 'USD'), (scaled_stoxx, 'EMEA', 'EUR')]:

    df = positions.reset_index().rename(
        columns={'RIC': 'internal_code', 0: 'target_notional'}
    ).assign(currency=currency)
    df.sort_values(by=['target_notional'], ascending=[False], inplace=True)
    df = df.reset_index(drop=True)
    
    send_new_portfolio(targets=df, region=region)


Found 0 error(s) while validating target_files/AMER/qrt_academy_ICL05_20260401-1022.csv
Found 0 error(s) while validating target_files/AMER/qrt_academy_ICL05_20260401-1022.csv
Reading private key from: /Users/dcunning/.ssh/icl05_id_rsa
Connecting to sftp.qrt.cloud:22
Logging in as q8576
Uploading target_files/AMER/qrt_academy_ICL05_20260401-1022.csv to incoming/amer/qrt_academy_ICL05_20260401-1022.csv
File 'target_files/AMER/qrt_academy_ICL05_20260401-1022.csv' successfully uploaded to AMER.
Found 0 error(s) while validating target_files/EMEA/qrt_academy_ICL05_20260401-1022.csv
Found 0 error(s) while validating target_files/EMEA/qrt_academy_ICL05_20260401-1022.csv
Reading private key from: /Users/dcunning/.ssh/icl05_id_rsa
Connecting to sftp.qrt.cloud:22
Logging in as q8576
Uploading target_files/EMEA/qrt_academy_ICL05_20260401-1022.csv to incoming/emea/qrt_academy_ICL05_20260401-1022.csv
File 'target_files/EMEA/qrt_academy_ICL05_20260401-1022.csv' successfully uploaded to EMEA.


### Factor Portfolio Optimisation

- combine ind mom, mom, value, portfolio optimisation
- benchmarks = [".SPX", ".STOXX50E"]

## Instructions

### Hedging
To hedge excessive residual beta in your equity portfolio:
1.	Compute the beta of each stock using a 250-day covariance with its benchmark:
$$\text{Beta} = 0.2 + 0.8 \times \frac{\text{Cov}_{250d}(\text{stock}, \text{benchmark})}{\text{Var}_{250d}(\text{benchmark})}$$
2. Hedge the beta exposure by taking a position in the benchmark equal to:
$$\text{Hedge Position} = -\text{Beta}$$
3. Benchmarks:
  - AMER: S&P 500 (SPX)
  - EMEA: Eurostoxx50

This means that if a stock has a beta of 1.2, you would short 1.2 units of the benchmark to neutralize the market exposure.

### Trading Limits
There are some trading limits for single stocks: 
- Max position = 2.5% of the 60 trading days ADV 
- Max traded per day = 2.5% of the 60 trading days ADV 

For single stocks, there is also a max position in term of USD: 
- Max position = 2 M USD 

If the implied trading by targets exceeds the above limits, the position will be kept constant until all limits are not exceeded anymore. 

### Trading Costs
The execution price will be the mid-price at the end of the next minute after we have received your file (without format error) on the ftp. Additional execution costs of 2bps also apply, along with 0.5% (annually) for financing. 

100% of dividends are paid on short positions. 70% of dividends are received on long positions. 

### Ex-Ante Portfolio Risk

Risk Formula
$$
\text{Risk}_t = \sqrt{ x_t^\top \, \Sigma_t \, x_t }
$$

Where:
- $x_t$ = portfolio weights at time $t$
- $\Sigma_t$ = covariance matrix of asset returns  

Computation Steps
1. Select last 60 daily returns.  
2. Compute daily P&L:

$$
\text{PnL}_t = \text{positions}_{t-1} \cdot \text{return}_t
$$

3. Compute standard deviation of P&L.  
4. Annualise: multiply by $\sqrt{252}$.  

Simulation Parameters
- Asset class: Equities  
- Regions: AMER, EMEA  
- Universe: Russell 3000, Stoxx 600  
- Trading hours: Standard per region  
- Execution cost: 2 bps  
- Financing cost: 0.5% GMV  
- Dividend tax: 30%  
- Spread cost: 0  
- Max traded/day: 2.5% ADV  
- Position limit: 2.5% ADV  
- Auto-hedging: TRUE  
- Risk limit: 500k USD (annualised)